In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from surveycodex import get_survey

from galgenai import get_device
from galgenai.config import load_config
from galgenai.data.cosmos_dataset import load_fits_dataset, make_loaders
from galgenai.data.normalization import (
    get_conditional_norm_fn,
    get_image_norm_fn,
)
from galgenai.models import VAE, ConditionalNormalizingFlow
from galgenai.utils import counts_to_magnitude

# 1. Set Configuration and Paths

In [ ]:
# Survey
hsc_survey = get_survey("HSC")
RUN_NAME = "arcsinh"

In [ ]:
# Set run name and paths
device = get_device()
print(f"Using device: {device}")

# Load config
cfg = load_config()
cosmos_cfg = cfg["cosmos"]
train_cfg = cfg["training"]
model_cfg = train_cfg["model"]
cnf_cfg = train_cfg["cnf"]
norm_cfg = cosmos_cfg["normalization"]

# Paths
output_dir = Path(train_cfg["output_dir"]) / RUN_NAME
vae_ckpt_path = output_dir / "vae" / "checkpoints" / "best.pt"
cnf_ckpt_path = output_dir / "cnf" / "checkpoints" / "best.pt"
norm_stats_path = output_dir / "norm_stats.yaml"
cond_stats_path = output_dir / "cond_stats.yaml"

# Create plots directory
plots_dir = output_dir / "plots"
plots_dir.mkdir(exist_ok=True)

# Dataset config
dataset_path = cosmos_cfg["hf_dataset_path"]
nx = train_cfg["nx"]
batch_size = 64
num_workers = cosmos_cfg["num_workers"]

# Model config
in_channels = model_cfg["in_channels"]
latent_dim = model_cfg["latent_dim"]
condition_cols = cnf_cfg["condition_cols"]
condition_dim = len(condition_cols)


print(f"Output directory: {output_dir}")
print(f"Plots directory: {plots_dir}")
print(f"Image size: {nx}x{nx}")
print(f"Latent dim: {latent_dim}")
print(f"Condition cols: {condition_cols}")

# 2. Load Models and Data

In [ ]:
# Load VAE
vae = VAE(
    in_channels=in_channels,
    latent_dim=latent_dim,
    input_size=nx,
)
vae_checkpoint = torch.load(vae_ckpt_path, map_location=device, weights_only=False)
vae.load_state_dict(vae_checkpoint["model_state_dict"])
vae = vae.to(device).eval()
print(f"Loaded VAE from: {vae_ckpt_path}")

# Load CNF
cnf = ConditionalNormalizingFlow(
    latent_dim=latent_dim,
    condition_dim=condition_dim,
    num_blocks=cnf_cfg["num_blocks"],
    hidden_dim=cnf_cfg["hidden_dim"],
)
cnf_checkpoint = torch.load(cnf_ckpt_path, map_location=device, weights_only=False)
cnf.load_state_dict(cnf_checkpoint["model_state_dict"])
cnf = cnf.to(device).eval()
print(f"Loaded CNF from: {cnf_ckpt_path}")

In [ ]:
# Load image normalization stats
image_norm_type = train_cfg["vae"]["norm_type"]
print(f"Image normalization type: {image_norm_type}")

image_norm_fn, image_denorm_fn, norm_stats = get_image_norm_fn(
    img_norm_type=image_norm_type,
    config=norm_cfg["image"],
    return_denorm=True,
)

# Load conditional normalization stats
conditional_norm_fn, conditional_denorm_fn, cond_stats = get_conditional_norm_fn(
    config=norm_cfg["conditions"],
    return_denorm=True,
)

In [ ]:
# Load dataset
print(f"Loading dataset from: {dataset_path}")

# Extract catalog columns config for consistency with training
catalog_cols = cosmos_cfg["catalog_columns"]
mag_cols = catalog_cols["mag_cols"]
redshift_col = catalog_cols["redshift_col"]

# Load with same parameters as training script to use cached Arrow dataset
dataset_raw = load_fits_dataset(
    dataset_path,
    metadata_file="metadata.csv",
    mag_cols=mag_cols,
    redshift_col=redshift_col,
    mag_sentinel=cosmos_cfg.get("mag_sentinel"),
    redshift_sentinel=cosmos_cfg.get("redshift_sentinel"),
)

# Create loaders
train_loader, val_loader, test_loader = make_loaders(
    dataset_raw,
    nx=nx,
    batch_size=batch_size,
    num_workers=num_workers,
    train_ratio=cosmos_cfg["train_ratio"],
    val_ratio=cosmos_cfg["val_ratio"],
    random_seed=cosmos_cfg["split_seed"],
    image_norm_fn=image_norm_fn,
    return_aux_data=True,
    condition_cols=condition_cols,
    conditional_norm_fn=conditional_norm_fn,
)

print("\nDataset loaded:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
if test_loader is not None:
    print(f"  Test batches: {len(test_loader)}")
else:
    print("  No test set (test_ratio is 0) will use val set for testing")
    test_loader = val_loader

# Generate Simulations

In [ ]:
n_samples = 50

In [ ]:
all_conditions_normed = []
for batch in test_loader:
    _, _, _, conditions_norm = batch
    conditions_norm = conditions_norm.to(device)
    all_conditions_normed.append(conditions_norm)
    if len(all_conditions_normed) * batch_size > n_samples:
        break

all_conditions_normed = torch.cat(
    all_conditions_normed, dim=0
)  # (N_total, condition_dim)

sampled_conditions_normed = all_conditions_normed[:n_samples]
sampled_conditions_denormed = conditional_denorm_fn(sampled_conditions_normed)

r_band_idx = condition_cols.index("mag_model_hsc-r")
mag_r_range = sampled_conditions_denormed[:, r_band_idx]

print(f"\nSampled {n_samples} real galaxy conditions")

In [ ]:
@torch.no_grad()
def generate_galaxies(cnf_model, vae_model, conditions, num_samples=1):
    """
    Generate galaxy images from conditional variables.

    Args:
        cnf_model: Trained CNF model
        vae_model: Trained VAE model
        conditions: Normalized conditioning variables (N, condition_dim)
        num_samples: Number of samples per condition

    Returns:
        Generated images (N, num_samples, in_channels, nx, nx)
    """
    cnf_model.eval()
    vae_model.eval()

    N = len(conditions)

    # Sample all latents in one batched call (much faster!)
    latents = cnf_model.sample(conditions, num_samples=num_samples)
    # Shape: (N * num_samples, latent_dim)

    # Decode all latents to images in one batched call
    images = vae_model.decoder(latents)

    # Reshape to (N, num_samples, in_channels, nx, nx)
    images = images.reshape(N, num_samples, in_channels, nx, nx)

    return images.cpu()


# Generate images
num_samples_per_condition = (
    25  # Generate multiple samples per condition to show variation
)
generated_images = generate_galaxies(
    cnf, vae, sampled_conditions_normed, num_samples=num_samples_per_condition
)

print(f"Generated images shape: {generated_images.shape}")

In [ ]:
# Denormalize images to physical units
denormalized_images = []

for i in range(generated_images.shape[0]):
    samples = []
    for j in range(generated_images.shape[1]):
        img_norm = generated_images[i, j]  # (in_channels, nx, nx)
        img_denorm = image_denorm_fn(img_norm)
        samples.append(img_denorm)
    denormalized_images.append(torch.stack(samples))

denormalized_images = torch.stack(denormalized_images)

# Results

## 1. Example Galaxy Simulations

In [ ]:
def display_galaxy_comparison(
    generated_img, catalog_mags, measured_mags, galaxy_idx, redshift, figsize=(20, 4)
):
    """
    Display galaxy images in all 5 bands with both catalog and measured magnitudes.

    Args:
        generated_img: Generated image tensor (5, 32, 32)
        catalog_mags: List of catalog magnitudes for each band
        measured_mags: List of measured magnitudes for each band
        galaxy_idx: Galaxy index number
        redshift: Redshift value
    """
    bands = hsc_survey.available_filters
    n_bands = len(bands)

    # Create subplots with black background
    fig, axes = plt.subplots(1, n_bands, figsize=figsize, facecolor="black")

    # Display each band
    for i, (ax, band, cat_mag, meas_mag) in enumerate(
        zip(axes, bands, catalog_mags, measured_mags)
    ):
        img = generated_img[i].numpy()

        # Set axes background to black
        ax.set_facecolor("black")

        # Display raw flux without scaling
        im = ax.imshow(img, origin="lower", cmap="viridis")

        # Title with band and both magnitudes (white text)
        title_text = f"{band}-band\n"
        title_text += f"catalog: {cat_mag:.2f}\n"
        title_text += f"measured: {meas_mag:.2f}"
        ax.set_title(title_text, fontsize=11, color="white")
        ax.axis("off")

        # Add colorbar with white text
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        # cbar.set_label('flux (e⁻/s)', fontsize=9, color='white')
        cbar.ax.yaxis.set_tick_params(color="white")
        plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")

    # Overall title with galaxy information (white text)
    title = f"Galaxy {galaxy_idx}  |  Redshift (z) = {redshift:.3f}"
    fig.suptitle(title, fontsize=15, y=1.02, color="white")

    plt.tight_layout()
    return fig

In [ ]:
# Plt and save example galaxy figures
# Sample 3 random galaxies from sampled conditions
n_examples = 3
example_indices = np.random.choice(n_samples, n_examples, replace=False)

# Get the sampled conditions for these 3 galaxies
example_conditions_tensor_normed = sampled_conditions_normed[example_indices]

# Generate images for these 3 galaxies
example_images_gen = generate_galaxies(
    cnf, vae, example_conditions_tensor_normed, num_samples=1
)

# Denormalize images
example_images_denorm = []
for i in range(example_images_gen.shape[0]):
    img_norm = example_images_gen[i, 0]  # Take first sample
    img_denorm = image_denorm_fn(img_norm)
    example_images_denorm.append(img_denorm)
example_images_denorm = torch.stack(example_images_denorm)


# Measure magnitudes
example_measured_mags = np.zeros((n_examples, in_channels))

for i in range(n_examples):
    for band_idx, band in enumerate(hsc_survey.available_filters):
        img = example_images_denorm[i, band_idx].numpy()
        total_flux = img.sum()
        measured_mag = counts_to_magnitude(total_flux, hsc_survey, band)
        example_measured_mags[i, band_idx] = measured_mag

# Plot and save
example_conditions_tensor_denormed = conditional_denorm_fn(
    example_conditions_tensor_normed
)
for i in range(n_examples):
    catalog_mags = example_conditions_tensor_denormed[i, :5]
    measured_mags = example_measured_mags[i]
    redshift = example_conditions_tensor_denormed[i, 5]

    fig = display_galaxy_comparison(
        example_images_denorm[i], catalog_mags, measured_mags, i + 1, redshift
    )

    fig.savefig(
        plots_dir / f"cnf_galaxy_example_{i + 1}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="black",
        pad_inches=0.3,
    )
    plt.show()

## Variation in sims

In [ ]:
num_samples_to_show = 3

# Pick one mag and show multiple samples

mag_idx = np.random.randint(n_samples)
mag_r_range = sampled_conditions_denormed[:, r_band_idx]
mag_value = mag_r_range[mag_idx]

fig, axes = plt.subplots(
    num_samples_to_show, in_channels, figsize=(15, 3 * num_samples_to_show)
)

if num_samples_to_show == 1:
    axes = axes[np.newaxis, :]

for sample_idx in range(num_samples_to_show):
    for band_idx, band_name in enumerate(hsc_survey.available_filters):
        ax = axes[sample_idx, band_idx]
        img = denormalized_images[mag_idx, sample_idx, band_idx].numpy()

        im = ax.imshow(img, cmap="viridis", origin="lower")
        ax.set_title(f"Sample {sample_idx + 1} - {band_name}-band", fontsize=10)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f"Multiple samples for r-band magnitude = {mag_value:.1f}\n"
    "(Shows stochastic variation in generation)",
    fontsize=14,
    y=1.0,
)
plt.tight_layout()
plt.show()

## 3. Quantitative Analysis: Flux vs Magnitude Per Band

In [ ]:
plevel_low = 15  # percentile level
plevel_high = 85

total_flux = np.zeros((n_samples, num_samples_per_condition, in_channels))

for i in range(n_samples):
    for j in range(num_samples_per_condition):
        for band_idx in range(in_channels):
            img = denormalized_images[i, j, band_idx].numpy()  # (nx, nx)
            total_flux[i, j, band_idx] = img.sum()

# Convert flux to magnitudes using HSC zero points
generated_mags = np.zeros_like(total_flux)

for band_idx, band in enumerate(hsc_survey.available_filters):
    # Convert flux to magnitude for each sample
    generated_mags[:, :, band_idx] = counts_to_magnitude(
        total_flux[:, :, band_idx], hsc_survey, band
    )

# Create individual plots for each band with black background
fig, axes = plt.subplots(
    1, len(hsc_survey.available_filters), figsize=(25, 5), facecolor="black"
)
fig.patch.set_facecolor("black")

for band_idx, band_name in enumerate(hsc_survey.available_filters):
    ax = axes[band_idx]
    ax.set_facecolor("black")

    # Get the input (catalog) magnitudes for this band from our sampled conditions
    input_mags = sampled_conditions_denormed[:, band_idx].cpu().numpy()

    # Get the generated magnitudes for this band
    gen_mags = generated_mags[:, :, band_idx]  # (n_samples, num_samples_per_condition)

    # Filter out NaN and inf values before computing statistics
    gen_mags_clean = gen_mags.copy()
    gen_mags_clean[~np.isfinite(gen_mags)] = np.nan

    # Calculate median and percentiles across samples (ignoring NaN)
    medians = np.nanmedian(gen_mags_clean, axis=1)  # (n_samples,)
    percentile_low = np.nanpercentile(gen_mags_clean, plevel_low, axis=1)
    percentile_high = np.nanpercentile(gen_mags_clean, plevel_high, axis=1)

    # Filter out any remaining NaN values in the aggregated statistics
    valid_mask = np.isfinite(medians) & np.isfinite(input_mags)
    input_mags_valid = input_mags[valid_mask]
    medians_valid = medians[valid_mask]
    plow_valid = percentile_low[valid_mask]
    phigh_valid = percentile_high[valid_mask]

    if len(input_mags_valid) == 0:
        ax.text(
            0.5,
            0.5,
            "No valid data",
            ha="center",
            va="center",
            transform=ax.transAxes,
            color="white",
        )
        ax.set_title(f"{band_name}-band", color="white")
        continue

    # Sort by input magnitude to avoid zigzag in shaded region
    sort_idx = np.argsort(input_mags_valid)
    input_mags_sorted = input_mags_valid[sort_idx]
    medians_sorted = medians_valid[sort_idx]
    plow_sorted = plow_valid[sort_idx]
    phigh_sorted = phigh_valid[sort_idx]

    # Plot shaded percentile region with brighter color for visibility
    ax.fill_between(
        input_mags_sorted,
        plow_sorted,
        phigh_sorted,
        alpha=0.4,
        color="cyan",
        label=f"{plevel_low}-{plevel_high}%",
    )

    # Scatter plot of median values with bright color
    ax.scatter(
        input_mags_valid, medians_valid, s=30, alpha=0.9, color="cyan", label="data"
    )

    # Plot ideal 1:1 line (perfect reconstruction)
    mag_range = np.array([input_mags_sorted.min(), input_mags_sorted.max()])
    ax.plot(mag_range, mag_range, "--", alpha=0.9, color="white", label="x=y", lw=2)

    ax.set_xlabel(f"{band_name}-band input magnitude", fontsize=12, color="white")
    ax.set_ylabel(f"{band_name}-band generated magnitude", fontsize=11, color="white")

    # Style the legend with white text
    legend = ax.legend(fontsize=9)
    legend.get_frame().set_facecolor("black")
    legend.get_frame().set_edgecolor("white")
    for text in legend.get_texts():
        text.set_color("white")

    ax.invert_xaxis()  # Brighter (lower mag) on the left
    ax.invert_yaxis()  # Brighter (lower mag) on the bottom
    ax.grid(True, alpha=0.3, color="white")

    # Set tick colors to white
    ax.tick_params(colors="white", which="both")
    for spine in ax.spines.values():
        spine.set_edgecolor("white")

fig.suptitle(
    f"Generated vs Input Magnitude ({num_samples_per_condition} samples per input for {n_samples} conditions)",
    fontsize=13,
    color="white",
)
plt.tight_layout()
plt.show()


# Save the figure
fig.savefig(
    plots_dir / "magnitude_comparison_all_bands.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="black",
    pad_inches=0.3,
)
plt.close(fig)